![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `ibm/granite-3-3-8b-instruct` to perform chat conversation with control roles

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook provides a detailed demonstration of the steps and code required to showcase support for control roles available in IBM Granite models.

Some familiarity with Python is helpful. This notebook uses Python 3.11.

## Learning goal

The purpose of this notebook is to demonstrate how to use control roles available in IBM Granite models.

## Table of Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Set up the Foundation Model on IBM watsonx.ai](#Set-up-the-Foundation-Model-on-IBM-watsonx.ai)
3. [Work with chat messages](#Work-with-chat-messages)
4. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install dependencies

**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U "ibm-watsonx-ai" | tail -n 1

### Define the watsonx.ai credentials
Use the code cell below to define the watsonx.ai credentials that are required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">Managing user API keys</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

### Define the project ID
You need to provide the project ID to give the Foundation Model the context for the call. If you have a default project ID set in Watson Studio, the notebook obtains that project ID. Otherwise, you need to provide the project ID in the code cell below.

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Enter your project_id and hit enter: ")

<a id="Set-up-the-Foundation-Model-on-IBM-watsonx.ai"></a>
## Set up the Foundation Model on IBM watsonx.ai


Specify the `model_id` of the model you will use for the chat with tools.

In [4]:
model_id = "ibm/granite-3-3-8b-instruct"

### Define the model parameters

You might need to adjust model parameters depending on the model you use.

In [5]:
from ibm_watsonx_ai.foundation_models.schema import TextChatParameters

TextChatParameters.show()

+-----------------------+----------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| PARAMETER             | TYPE                                   | EXAMPLE VALUE                                                                                                                                                                                                                                                                   |
+=======================+========================================+============================================================================================================================================================================================================================================================

In [6]:
params = TextChatParameters(temperature=1)

### Initialize the model

Initialize the `ModelInference` class with the previously set parameters.

In [7]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(
    model_id=model_id, credentials=credentials, project_id=project_id, params=params
)

<a id="Work-with-chat-messages"></a>
## Work with chat messages

### Work with a simple chat message using `chat_stream`

In [8]:
messages = [{"role": "user", "content": "Which Formula 1 team is partnered with IBM?"}]

chat_stream_response = model.chat_stream(messages=messages)

In [9]:
for chunk in chat_stream_response:
    if chunk["choices"]:
        print(chunk["choices"][0]["delta"].get("content", ""), end="", flush=True)

Scuderia Ferrari is the Formula 1 racing team partnered with IBM.

### Work with chat message with `thinking` control role
The `thinking` control role enables enhanced reasoning in model responses.

In [10]:
messages = [
    {"role": "control", "content": "thinking"},
    {"role": "user", "content": "Tell me a joke"},
]

chat_stream_response = model.chat_stream(messages=messages)

In [11]:
for chunk in chat_stream_response:
    if chunk["choices"]:
        print(chunk["choices"][0]["delta"].get("content", ""), end="", flush=True)

<think>Undoubtedly, the type of joke would greatly depend on personal preferences. For a classic, broadly appealing one-liner would be suitable. Here’s an example that's light-hearted and related to an often-discussed topic - AI:

Why was the computer cold? It left its Windows open.

This joke plays on the double meaning of "left its Windows open" - both referring to a computer system vulnerability and literal means of cooling.</think><response>### Joke:
Why was the computer cold? It left its Windows open.

### Explanation:
The humor in this joke stems from a play on words. "Windows" can refer both to the computer operating system (a specific brand, here Microsoft Windows) and to the physical openings through which cool air might enter a room. It's a simple, technology-based joke that many might find relatable given the commonality of such systems.</response>

### Work with chat message with `length` control role
Using the `length` control role, you can specify how long the model response should be.

#### Short length

In [12]:
messages = [
    {"role": "control", "content": "length short"},
    {"role": "user", "content": "Explain torque"},
]

chat_stream_response = model.chat_stream(messages=messages)

In [13]:
for chunk in chat_stream_response:
    if chunk["choices"]:
        print(chunk["choices"][0]["delta"].get("content", ""), end="", flush=True)

Torque is a measure of force that causes rotation around an axis. It's the rotation equivalent of linear force, calculated as the product of force and lever arm (distance from the axis), represented by the equation: Torque (τ) = Force (F) × Distance (r).

#### Long length

In [14]:
messages = [
    {"role": "control", "content": "length long"},
    {"role": "user", "content": "Explain torque"},
]

chat_stream_response = model.chat_stream(messages=messages)

In [15]:
for chunk in chat_stream_response:
    if chunk["choices"]:
        print(chunk["choices"][0]["delta"].get("content", ""), end="", flush=True)

Torque, also known as the moment of force, is a fundamental concept in physics and engineering that describes the rotational effect of a force applied to an object about a specific axis. It's a measure of the force's tendency to cause an object to rotate or turn around that axis. 

Here are the key aspects of torque:

1. **Definition**: Torque (τ) is the rotational equivalent of linear force (F). Just as force causes linear acceleration (F = ma), torque causes angular acceleration (τ = Iα), where 'I' is the moment of inertia and 'α' is the angular acceleration of the body.

2. **Formula**: The formula for torque around an axis is:

   τ = r x F

   - Here, 'r' represents the vector from the axis of rotation to the point of force application, usually expressed in meters (m).
   - 'F' is the force vector acting on the body, also in meters (m), and 'x' denotes the cross product of the two vectors. The cross product results in a vector that's perpendicular to both the force and displacemen

### Work with chat message with `originality` control role
The `originality` control role specifies the type of summarization of the model response.

#### Extractive originality
When using `extractive` option, the model will create a summary of a given topic using the exact words used in its input data and may also cite its references.

In [16]:
messages = [
    {"role": "control", "content": "originality extractive"},
    {"role": "user", "content": "Mineral composition of granites"},
]

chat_stream_response = model.chat_stream(messages=messages)

In [17]:
for chunk in chat_stream_response:
    if chunk["choices"]:
        print(chunk["choices"][0]["delta"].get("content", ""), end="", flush=True)

Granites are igneous rocks composed mainly of quartz, feldspar, and mica minerals. Here's a breakdown of their typical mineral composition:

1. **Quartz (SiO2)**: Granites are primarily silica-rich rocks, with quartz making up about 20-40% of the rock. It's often the largest mineral and is colorless, transparent, or translucent.

2. **Feldspar**: This mineral group consists of plagioclase feldspar (darker, typically black or gray in fresh samples) and alkali feldspar (lighter, often white or pink). Feldspar makes up about 55-70% of granite. Potassium feldspar (orthoclase or microcline) is the most common type in granite.

3. **Mica**: These are sheet silicate minerals that include biotite (dark, usually black) and muscovite (light, typically white or pinkish). Mica comprises about 5-15% of granite.

4. **Amphiboles**: These minerals, such as hornblende, are often present in smaller amounts (up to 10%). They give granite a darker hue when present.

5. **Pyroxenes and Olivine**: These mi

#### Abstractive originality
When using `abstractive` originality, the model will paraphrase its data in order to create its response.

In [18]:
messages = [
    {"role": "control", "content": "originality abstractive"},
    {"role": "user", "content": "Mineral composition of granites"},
]

chat_stream_response = model.chat_stream(messages=messages)

In [19]:
for chunk in chat_stream_response:
    if chunk["choices"]:
        print(chunk["choices"][0]["delta"].get("content", ""), end="", flush=True)

Granites are igneous rocks primarily composed of quartz, feldspar, and mica minerals. Here's a breakdown of their typical mineral composition:

1. **Quartz (SiO2)**: Quartz is usually the most abundant mineral in granites, making up about 20-40% of the rock. It's a hard, colorless, or white mineral.

2. **Feldspars (K-feldspar and Plagioclase feldspar)**: Feldspars constitute around 40-70% of granite. They can be either alkali feldspar (rich in potassium, orthoclase) or plagioclase feldspar (a series of rocks ranging from calcium-rich to sodium-rich).

3. **Mica (Muscovite and Biotite)**: Micas make up 5-15% of granite. They are platey minerals that give granite its characteristic shiny appearance. Muscovite mica is light-colored, while biotite mica is dark-colored.

4. **Biotite**: This is a common black mica in granites, accounting for about 3-8%.

5. **Garnet**: Some granites may contain small amounts of garnet (2-5%), contributing red or pink color to the rock.

6. **Amplhibole**: 

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to work with chat models using tools and watsonx.ai.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Author

**Rafał Chrzanowski**, Software Engineer Intern at watsonx.ai.

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.